# Industrial PPE Detection Training & ONNX Export (Foreman Pipeline)
This notebook trains a YOLOv8s model on an industrial safety dataset (workers, hardhats, vests)
and exports the weights to PyTorch (`.pt`) and ONNX (`.onnx`) for edge deployment.

In [ ]:
# Step 1: Install Ultralytics and dependencies
!pip install ultralytics roboflow -q

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 2: Download the PPE Dataset from Roboflow
from roboflow import Roboflow

# Note: If you are logged into Roboflow, it auto-fills your real API key.
# Otherwise, get your free key from Roboflow -> Settings -> Roboflow API
rf = Roboflow(api_key="YOUR_KEY")
project = rf.workspace("hx-hezqh").project("ppe-detection-yfmym")
version = project.version(1)
dataset = version.download("yolov8")

print(f"Dataset downloaded successfully to: {dataset.location}")

In [ ]:
# Step 3: Fine-Tune YOLOv8s on GPU
from ultralytics import YOLO

# Load pretrained YOLOv8s base model
model = YOLO('yolov8s.pt')

# Train for 15-20 epochs (takes ~12-15 mins on Kaggle GPU)
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=15,
    imgsz=640,
    batch=16,
    device=0
)

In [ ]:
# Step 4: Validate and print mAP metrics
metrics = model.val()
print(f"Validation mAP@50: {metrics.box.map50:.3f}")
print(f"Validation mAP@50-95: {metrics.box.map:.3f}")

In [ ]:
# Step 5: Export to ONNX (The Edge Format)
# This produces the exact .onnx engine used in the edge vision pipeline
success = model.export(format='onnx', dynamic=False, opset=17)
print(f"Exported ONNX Model: {success}")

In [ ]:
# Step 6: Download your weights to put into your local repo!
# Files will be saved in runs/detect/train/weights/best.pt and best.onnx
from IPython.display import FileLink
print("Download these files to place into your repo's 'models/' folder:")
print("1. best.pt")
print("2. best.onnx")